In [1]:
import numpy as np
import pandas as pd

# Load data
file_path = "data_kuesioner_kantin.csv"
df_raw = pd.read_csv(file_path)

# Tampilkan 5 baris pertama data
print("Total Data Responden:", len(df_raw))
df_raw.head()

Total Data Responden: 37


,Nama,Fakultas,Angkatan,Kantin Fakultas,Harga,Waktu Tunggu,Kualitas,Kesesuaian Porsi,Kebersihan Meja,Kursi Jam Sibuk,Fasilitas,Pelayanan
0,Rafi Fajri Ramadhani,Fakultas Ilmu Komputer,2024,Kantin MM,20000,5,5,5,10,60,4,5
1,Ahmad Rafly,Fakultas Ilmu Komputer,2024,Kantin FH,15000,10,4,4,10,60,3,5
2,Reva,Politeknik Sriwijaya,2024,Kantin Polsri,15000,15,4,4,5,80,4,4
3,M. Farrel Alfasya,Fakultas Ekonomi,2024,Kantin MM,20000,5,4,4,10,20,5,5
4,M RafaIhtisan,Fakultas Ilmu Komputer,2024,Kantin Fasilkom,15000,5,4,4,10,60,4,5


### **Step 1 - Agregasi Matriks Keputusan ($X_{ij}$)**

In [3]:
# Tentukan nama kolom kantin dan 8 kriteria yang digunakan
kolom_kantin = "Kantin Fakultas"
kolom_kriteria = [
    "Harga",
    "Waktu Tunggu",
    "Kualitas",
    "Kesesuaian Porsi",
    "Kebersihan Meja",
    "Kursi Jam Sibuk",
    "Fasilitas",
    "Pelayanan",
]

# Menghitung nilai rata-rata penilaian kuesioner per kantin (Matriks Keputusan 7 x 8)
matriks_keputusan = df_raw.groupby(kolom_kantin)[kolom_kriteria].mean()

# Tampilkan Matriks Keputusan
matriks_keputusan

,Harga,Waktu Tunggu,Kualitas,Kesesuaian Porsi,Kebersihan Meja,Kursi Jam Sibuk,Fasilitas,Pelayanan
Kantin Fakultas,,,,,,,,
Kantin D3 FE,15000.000000,5.000000,4.000000,5.000000,5.000000,40.000000,3.000000,4.000000
Kantin FH,16250.000000,8.750000,3.500000,3.750000,7.500000,45.000000,3.500000,4.250000
Kantin Fasilkom,14500.000000,8.500000,3.400000,3.100000,11.500000,52.000000,2.600000,3.600000
Kantin MM,19166.666667,6.666667,4.166667,4.166667,9.166667,56.666667,4.000000,4.666667
Kantin Pantai,15000.000000,10.000000,3.000000,3.333333,6.666667,33.333333,2.333333,3.333333
Kantin Polsri,14166.666667,11.666667,4.166667,3.833333,11.666667,43.333333,3.333333,3.833333
Kantin S1 FE,17500.000000,15.000000,4.333333,4.666667,18.333333,46.666667,2.833333,4.500000


### **Step 2 - Sifat, Bobot, & Average Solution ($AV_j$)**

In [4]:
# Setup sifat kriteria ('cost' atau 'benefit')
sifat_kriteria = {
    "Harga": "cost",
    "Waktu Tunggu": "cost",
    "Kualitas": "benefit",
    "Kesesuaian Porsi": "benefit",
    "Kebersihan Meja": "cost",
    "Kursi Jam Sibuk": "benefit",
    "Fasilitas": "benefit",
    "Pelayanan": "benefit",
}

# Setup bobot kriteria (total = 1.0)
bobot = np.array([0.15, 0.10, 0.20, 0.15, 0.10, 0.10, 0.10, 0.10])

# Menghitung Nilai Rata-Rata Solusi (Average Solution / AV)
av = matriks_keputusan.mean(axis=0)

# Tampilkan nilai AV per kriteria
pd.DataFrame([av], index=["Average Solution (AV)"])

,Harga,Waktu Tunggu,Kualitas,Kesesuaian Porsi,Kebersihan Meja,Kursi Jam Sibuk,Fasilitas,Pelayanan
Average Solution (AV),15940.47619,9.369048,3.795238,3.978571,9.97619,45.285714,3.085714,4.02619


### **Step 3 & Step 4 - Matriks PDA & NDA**

In [5]:
# Inisialisasi DataFrame kosong untuk PDA dan NDA
pda = pd.DataFrame(0.0, index=matriks_keputusan.index, columns=kolom_kriteria)
nda = pd.DataFrame(0.0, index=matriks_keputusan.index, columns=kolom_kriteria)

# Perhitungan jarak deviasi positif (PDA) dan negatif (NDA)
for col in kolom_kriteria:
    av_j = av[col]
    x_j = matriks_keputusan[col]

    if sifat_kriteria[col] == "benefit":
        pda[col] = np.maximum(0, (x_j - av_j) / av_j)
        nda[col] = np.maximum(0, (av_j - x_j) / av_j)
    else:  # Kriteria 'cost'
        pda[col] = np.maximum(0, (av_j - x_j) / av_j)
        nda[col] = np.maximum(0, (x_j - av_j) / av_j)

print("--- MATRIKS PDA (Positive Distance from Average) ---")
display(pda)

print("\n--- MATRIKS NDA (Negative Distance from Average) ---")
display(nda)

--- MATRIKS PDA (Positive Distance from Average) ---


,Harga,Waktu Tunggu,Kualitas,Kesesuaian Porsi,Kebersihan Meja,Kursi Jam Sibuk,Fasilitas,Pelayanan
Kantin Fakultas,,,,,,,,
Kantin D3 FE,0.058999,0.466328,0.053952,0.256732,0.498807,0.000000,0.000000,0.000000
Kantin FH,0.000000,0.066074,0.000000,0.000000,0.248210,0.000000,0.134259,0.055588
Kantin Fasilkom,0.090366,0.092757,0.000000,0.000000,0.000000,0.148265,0.000000,0.000000
Kantin MM,0.000000,0.288437,0.097867,0.047277,0.081146,0.251314,0.296296,0.159077
Kantin Pantai,0.058999,0.000000,0.000000,0.000000,0.331742,0.000000,0.000000,0.000000
Kantin Polsri,0.111277,0.000000,0.097867,0.000000,0.000000,0.000000,0.080247,0.000000
Kantin S1 FE,0.000000,0.000000,0.141782,0.172950,0.000000,0.030494,0.000000,0.117682



--- MATRIKS NDA (Negative Distance from Average) ---


,Harga,Waktu Tunggu,Kualitas,Kesesuaian Porsi,Kebersihan Meja,Kursi Jam Sibuk,Fasilitas,Pelayanan
Kantin Fakultas,,,,,,,,
Kantin D3 FE,0.000000,0.000000,0.000000,0.000000,0.000000,0.116719,0.027778,0.006505
Kantin FH,0.019417,0.000000,0.077792,0.057451,0.000000,0.006309,0.000000,0.000000
Kantin Fasilkom,0.000000,0.000000,0.104141,0.220826,0.152745,0.000000,0.157407,0.105855
Kantin MM,0.202390,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Kantin Pantai,0.000000,0.067344,0.209536,0.162178,0.000000,0.263933,0.243827,0.172088
Kantin Polsri,0.000000,0.245235,0.000000,0.036505,0.169451,0.043113,0.000000,0.047901
Kantin S1 FE,0.097834,0.601017,0.000000,0.000000,0.837709,0.000000,0.081790,0.000000


### **Step 5 - Total Terbobot SP & SN**

In [6]:
# Menghitung perkalian matriks terbobot (SUMPRODUCT di Excel)
sp = pda.dot(bobot)
sn = nda.dot(bobot)

# Tampilkan ringkasan nilai SP dan SN
df_sp_sn = pd.DataFrame(
    {"SP (Jumlah Terbobot PDA)": sp, "SN (Jumlah Terbobot NDA)": sn}
)
df_sp_sn

,SP (Jumlah Terbobot PDA),SN (Jumlah Terbobot NDA)
Kantin Fakultas,,
Kantin D3 FE,0.154664,0.015100
Kantin FH,0.050413,0.027719
Kantin Fasilkom,0.037657,0.095553
Kantin MM,0.134292,0.030358
Kantin Pantai,0.042024,0.140953
Kantin Polsri,0.044290,0.056046
Kantin S1 FE,0.069116,0.166727


### **Step 6 & 7 - Normalisasi (NSP/NSN), Skor Akhir (AS), & Ranking**

In [8]:
# Normalisasi NSP dan NSN
nsp = sp / sp.max()
nsn = 1 - (sn / sn.max())

# Menghitung Assessment Score (AS)
as_score = (nsp + nsn) / 2

# Menyusun Tabel Rekapitulasi Akhir
df_hasil = pd.DataFrame(
    {"SP": sp, "SN": sn, "NSP": nsp, "NSN": nsn, "Skor Akhir (AS)": as_score}
)

# Menentukan Perangkingan (Rank 1 = Skor AS tertinggi)
df_hasil["Ranking"] = df_hasil["Skor Akhir (AS)"].rank(
    ascending=False, method="min"
).astype(int)

# Urutkan berdasarkan Ranking terbaik
df_hasil = df_hasil.sort_values(by="Ranking")

print("=== HASIL AKHIR PERANGKINGAN METODE EDAS ===")
df_hasil

=== HASIL AKHIR PERANGKINGAN METODE EDAS ===


,SP,SN,NSP,NSN,Skor Akhir (AS),Ranking
Kantin Fakultas,,,,,,
Kantin D3 FE,0.154664,0.015100,1.000000,0.909431,0.954716,1
Kantin MM,0.134292,0.030358,0.868284,0.817915,0.843100,2
Kantin FH,0.050413,0.027719,0.325953,0.833743,0.579848,3
Kantin Polsri,0.044290,0.056046,0.286361,0.663847,0.475104,4
Kantin Fasilkom,0.037657,0.095553,0.243477,0.426891,0.335184,5
Kantin S1 FE,0.069116,0.166727,0.446883,0.000000,0.223441,6
Kantin Pantai,0.042024,0.140953,0.271713,0.154586,0.213149,7
